In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import cv2
import os
from tqdm import tqdm

ROOT = "/content/drive/MyDrive/cow/2/dataset"
VIDEO_ROOT = f"{ROOT}/videos"
FRAME_ROOT = f"{ROOT}/frames"

TARGET_FPS = 5                 # sufficient for gait analysis
LABELS = ["lame", "non_lame"]  # use both classes

for label in LABELS:
    video_dir = f"{VIDEO_ROOT}/{label}"
    frame_dir = f"{FRAME_ROOT}/{label}"
    os.makedirs(frame_dir, exist_ok=True)

    videos = [v for v in os.listdir(video_dir) if v.endswith(".mp4")]

    for video in tqdm(videos, desc=f"Processing {label}"):
        cap = cv2.VideoCapture(f"{video_dir}/{video}")
        fps = cap.get(cv2.CAP_PROP_FPS)
        interval = max(1, int(fps / TARGET_FPS))

        out_dir = f"{frame_dir}/{video.replace('.mp4','')}"
        os.makedirs(out_dir, exist_ok=True)

        frame_id, save_id = 0, 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_id % interval == 0:
                cv2.imwrite(f"{out_dir}/frame_{save_id:04d}.jpg", frame)
                save_id += 1

            frame_id += 1

        cap.release()


In [ ]:
!pip install ultralytics


In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")  # small & fast


In [ ]:
import os, cv2, csv

ROOT = "/content/drive/MyDrive/cow/2/dataset"
FRAME_ROOT = f"{ROOT}/frames"
DET_ROOT = f"{ROOT}/detections"
os.makedirs(DET_ROOT, exist_ok=True)

LABELS = ["lame", "non_lame"]
COW_CLASS_ID = 19  # COCO class id for cow
CONF_THRES = 0.4


In [ ]:
for label in LABELS:
    frame_label_dir = f"{FRAME_ROOT}/{label}"
    det_label_dir = f"{DET_ROOT}/{label}"
    os.makedirs(det_label_dir, exist_ok=True)

    videos = os.listdir(frame_label_dir)

    for vid in videos:
        frame_dir = f"{frame_label_dir}/{vid}"
        csv_path = f"{det_label_dir}/{vid}.csv"

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["frame", "x1", "y1", "x2", "y2", "conf"])

            for frame_name in sorted(os.listdir(frame_dir)):
                img = cv2.imread(f"{frame_dir}/{frame_name}")
                res = model(img, conf=CONF_THRES, verbose=False)[0]

                # keep the largest cow box (typical: one cow/video)
                best = None
                best_area = 0

                for b in res.boxes:
                    if int(b.cls) == COW_CLASS_ID:
                        x1,y1,x2,y2 = map(float, b.xyxy[0])
                        area = (x2-x1)*(y2-y1)
                        if area > best_area:
                            best_area = area
                            best = (x1,y1,x2,y2,float(b.conf))

                if best:
                    writer.writerow([frame_name, *best])


In [ ]:
import os
import cv2
import pandas as pd
from tqdm import tqdm
from google.colab import drive

# ------------------------
# 3. Paths (adjust to your Drive)
# ------------------------
FRAMES_ROOT = "/content/drive/MyDrive/cow/2/dataset/frames"     # extracted frames
CSV_ROOT    = "/content/drive/MyDrive/cow/2/dataset/detections" # YOLO CSVs
OUT_ROOT    = "/content/drive/MyDrive/cow/2/dataset/cow_crop"  # output crops

os.makedirs(OUT_ROOT, exist_ok=True)

# ------------------------
# 4. Labels
# ------------------------
labels = ["non_lame", "lame"]

# ------------------------
# 5. Loop over labels, videos, and CSVs
# ------------------------
for label in labels:
    frame_label_dir = os.path.join(FRAMES_ROOT, label)
    csv_label_dir    = os.path.join(CSV_ROOT, label)
    out_label_dir   = os.path.join(OUT_ROOT, label)
    os.makedirs(out_label_dir, exist_ok=True)

    videos = os.listdir(frame_label_dir)

    for video in videos:
        frame_dir = os.path.join(frame_label_dir, video)
        csv_file  = os.path.join(csv_label_dir, f"{video}.csv")
        out_dir   = os.path.join(out_label_dir, video)
        os.makedirs(out_dir, exist_ok=True)

        if not os.path.exists(csv_file):
            print(f"CSV not found for {video}, skipping...")
            continue

        df = pd.read_csv(csv_file)

        # optional: filter weak detections
        df = df[df["conf"] >= 0.01].reset_index(drop=True)

        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Cropping {video}"):
            img_path = os.path.join(frame_dir, row["frame"])
            img = cv2.imread(img_path)

            if img is None:
                continue

            h, w, _ = img.shape
            x1, y1 = max(0, int(row["x1"])), max(0, int(row["y1"]))
            x2, y2 = min(w, int(row["x2"])), min(h, int(row["y2"]))

            crop = img[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            # Modify save_path to include video name
            save_filename = f"{video}_{row['frame']}"
            save_path = os.path.join(out_dir, save_filename)
            cv2.imwrite(save_path, crop)

print("✅ Cow cropping completed and saved to Google Drive.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
lame_xml_path = "/content/drive/MyDrive/cow/2/dataset/annotations/lame.xml"
non_lame_xml_path = "/content/drive/MyDrive/cow/2/dataset/annotations/non_lame.xml"

output_dir = "/content/drive/MyDrive/cow/2/dataset/annotations/"


In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

def cvat_xml_to_pose_csv(xml_path, label, output_csv):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    rows = []

    for image in root.findall("image"):
        img_name = image.attrib["name"]
        img_w = float(image.attrib["width"])
        img_h = float(image.attrib["height"])

        points_tag = image.find("points")
        if points_tag is None:
            continue

        points = points_tag.attrib["points"].split(";")
        if len(points) != 17:
            print(f"Skipping {img_name}: {len(points)} points")
            continue

        row = {"image": img_name}

        for i, p in enumerate(points):
            x, y = map(float, p.split(","))
            row[f"kp{i}_x"] = x / img_w
            row[f"kp{i}_y"] = y / img_h

        row["label"] = label
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"Saved {len(df)} samples → {output_csv}")

# Convert both classes
cvat_xml_to_pose_csv(
    lame_xml_path,
    label=1,
    output_csv=output_dir + "pose_lame.csv"
)

cvat_xml_to_pose_csv(
    non_lame_xml_path,
    label=0,
    output_csv=output_dir + "pose_non_lame.csv"
)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- Configuration ---

# Paths
FRAMES_ROOT = "/content/drive/MyDrive/cow/2/dataset/cow_crop" # Root for cropped cow images
CSV_ROOT    = "/content/drive/MyDrive/cow/2/dataset/annotations" # Root for keypoint CSVs
OUT_ROOT    = "/content/drive/MyDrive/cow/2/dataset/cow_crop_augmented" # Output root for augmented images
OUTPUT_CSV_DIR = "/content/drive/MyDrive/cow/2/dataset/annotations_augmented" # Output root for augmented CSVs

# Percentage of existing samples to augment (0.0 to 1.0)
AUGMENTATION_PERCENTAGE = 1

# Keypoint Swapping Map:
# Define a dictionary where keys are original keypoint indices and values are the indices
# of their symmetrical counterparts after a horizontal flip.
# Based on the provided keypoint names (0-indexed):
# kp5 (FL_upper) <-> kp8 (FR_upper)
# kp6 (FL_knee)  <-> kp9 (FR_knee)
# kp7 (FL_hoof)  <-> kp10 (FR_hoof)
# kp11 (HL_upper) <-> kp14 (HR_upper)
# kp12 (HL_knee)  <-> kp15 (HR_knee)
# kp13 (HL_hoof)  <-> kp16 (HR_hoof)
KP_SWAP_MAP = {
    5: 8,  8: 5,   # FL_upper <-> FR_upper
    6: 9,  9: 6,   # FL_knee <-> FR_knee
    7: 10, 10: 7,  # FL_hoof <-> FR_hoof
    11: 14, 14: 11, # HL_upper <-> HR_upper
    12: 15, 15: 12, # HL_knee <-> HR_knee
    13: 16, 16: 13  # HL_hoof <-> HR_hoof
}

os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(OUTPUT_CSV_DIR, exist_ok=True)

# --- Helper Functions ---

def flip_image(img_path, output_img_path):
    """Loads an image, flips it horizontally, and saves it."""
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: Could not load image {img_path}, skipping flip.")
        return None
    flipped_img = cv2.flip(img, 1) # Flip horizontally
    cv2.imwrite(output_img_path, flipped_img)
    return flipped_img.shape[1], flipped_img.shape[0] # Return width, height

def augment_keypoints(row, kp_swap_map, original_df_columns):
    """Applies horizontal flip to keypoint coordinates and swaps symmetrical pairs."""
    new_row = row.copy()

    # First, apply the x-coordinate flip to all keypoints
    # Make a copy of the x-coordinates to avoid modifying while reading for swaps
    original_flipped_x_coords = {f"kp{i}_x": (1.0 - new_row[f"kp{i}_x"]) for i in range(17) if f"kp{i}_x" in new_row.index}

    # Apply swaps using the *already flipped* x-coordinates
    # Create a temporary storage for swapped values to avoid overwriting issues
    temp_kp_values = {}
    for i in range(17): # Populate temporary values with the initial flipped coordinates
        if f"kp{i}_x" in new_row.index:
            temp_kp_values[f"kp{i}_x"] = original_flipped_x_coords[f"kp{i}_x"]
            temp_kp_values[f"kp{i}_y"] = new_row[f"kp{i}_y"]

    # Now, perform the swaps
    for original_idx, swap_idx in kp_swap_map.items():
        if f"kp{original_idx}_x" in new_row.index and f"kp{swap_idx}_x" in new_row.index:
            # Swap x coordinates
            new_row[f"kp{original_idx}_x"] = temp_kp_values[f"kp{swap_idx}_x"]
            new_row[f"kp{swap_idx}_x"] = temp_kp_values[f"kp{original_idx}_x"]
            # Swap y coordinates (y-coordinates don't change with horizontal flip, but their association does)
            new_row[f"kp{original_idx}_y"] = temp_kp_values[f"kp{swap_idx}_y"]
            new_row[f"kp{swap_idx}_y"] = temp_kp_values[f"kp{original_idx}_y"]

    # For keypoints not involved in swapping, ensure their x-coordinates are flipped
    for i in range(17):
        if f"kp{i}_x" in new_row.index and i not in kp_swap_map:
            new_row[f"kp{i}_x"] = original_flipped_x_coords[f"kp{i}_x"]

    return new_row

print("Setup complete. Ready to augment data.")

# Task
Augment the cow dataset by horizontally flipping a subset of cropped cow images and their corresponding keypoint annotations. This involves loading the original `pose_lame.csv` and `pose_non_lame.csv` files, iterating through the samples, applying horizontal flips to images and adjusting keypoints according to the `KP_SWAP_MAP`, and then saving the augmented images to `/content/drive/MyDrive/cow/2/dataset/cow_crop_augmented` and combined (original + augmented) keypoint data to new CSVs in `/content/drive/MyDrive/cow/2/dataset/annotations_augmented`. The task concludes by confirming the successful completion of the data augmentation process.

## Load Original Pose CSVs

### Subtask:
Load the original `pose_lame.csv` and `pose_non_lame.csv` into pandas DataFrames. These contain the keypoint annotations for the cropped cow images.


**Reasoning**:
Load the `pose_lame.csv` and `pose_non_lame.csv` files into pandas DataFrames as requested in the subtask.



In [ ]:
lame_csv_path = os.path.join(output_dir, "pose_lame.csv")
non_lame_csv_path = os.path.join(output_dir, "pose_non_lame.csv")

df_lame_original = pd.read_csv(lame_csv_path)
df_non_lame_original = pd.read_csv(non_lame_csv_path)

print("Loaded df_lame_original:")
print(df_lame_original.head())
print("\nLoaded df_non_lame_original:")
print(df_non_lame_original.head())

**Reasoning**:
The next step is to combine the two original DataFrames (`df_lame_original` and `df_non_lame_original`) into a single DataFrame to process all samples uniformly for augmentation. This ensures that the augmentation logic can be applied across both classes seamlessly.



In [ ]:
df_original = pd.concat([df_lame_original, df_non_lame_original]).reset_index(drop=True)
print("Combined original DataFrame (df_original) created:")
print(df_original.head())

In [ ]:
from sklearn.model_selection import train_test_split

# Split the original combined DataFrame into training and testing sets
# The test set (20%) will be held out and not augmented
# The training set (80%) will be used for augmentation
df_train_original, df_test_holdout = train_test_split(
    df_original, test_size=0.2, random_state=42, stratify=df_original['label']
)

print(f"Original data split: {len(df_train_original)} samples for training, {len(df_test_holdout)} samples for testing (holdout).")
print("Training samples head (df_train_original):")
print(df_train_original.head())
print("\nTesting samples head (df_test_holdout):")
print(df_test_holdout.head())

**Reasoning**:
Now that the original data has been split into training and holdout test sets, the augmentation should only be applied to the training set. Therefore, `df_to_augment` will now be derived from `df_train_original` instead of `df_original`.

**Reasoning**:
Now that the original DataFrames are combined, the next step is to select a subset of these samples for augmentation based on the `AUGMENTATION_PERCENTAGE`. This involves randomly sampling rows from the combined DataFrame.



In [ ]:
num_samples_to_augment = int(len(df_original) * AUGMENTATION_PERCENTAGE)
df_to_augment = df_original.sample(n=num_samples_to_augment, random_state=42).copy()

print(f"Selected {len(df_to_augment)} samples ({AUGMENTATION_PERCENTAGE*100:.0f}% of total) for augmentation.")
print("Augmentation samples head:")
print(df_to_augment.head())

**Reasoning**:
Now, iterate through the samples selected for augmentation. For each sample, construct the image path, flip the image horizontally, and apply the keypoint augmentation function to update the keypoint coordinates. Save the flipped image to the designated output directory and store the augmented keypoint data in a list.



**Reasoning**:
The previous code failed because the `original_image_path` was incorrectly constructed, leading to `cv2.imread` failing to load the image and `flip_image` returning `None`. The `video_name` was extracted incorrectly from the `original_image_name`. The `original_image_name` has the format `video_folder_name_frame_XXXX.jpg`, and the `video_folder_name` should be extracted as the part before `_frame_`.



**Reasoning**:
The previous code failed because `cv2.imread` returned `None` for some image paths, leading to a `TypeError` when attempting to unpack `None`. This issue is likely due to the corresponding cropped image files not existing on disk, possibly because they were skipped during the initial cropping phase if the original frame could not be read or the crop was empty. To fix this, I will add an explicit check for the existence of the `original_image_path` before attempting to read and flip the image. If the file does not exist, a warning will be printed, and the sample will be skipped for augmentation, preventing the `TypeError`.



In [ ]:
augmented_data = []

for index, row in tqdm(df_to_augment.iterrows(), total=len(df_to_augment), desc="Augmenting images and keypoints"):
    original_image_name = row["image"]

    # Correctly extract video_folder_name from original_image_name
    # The image name format is typically 'video_id_frame_XXXX.jpg'
    # We need the 'video_id' part.
    frame_part_start_idx = original_image_name.rfind('_frame_')
    if frame_part_start_idx != -1:
        video_folder_name = original_image_name[:frame_part_start_idx]
    else:
        # Fallback if the naming convention is unexpected, though it should be consistent.
        print(f"Warning: Image name '{original_image_name}' does not contain '_frame_'. Attempting fallback extraction.")
        video_folder_name = '_'.join(original_image_name.split('_')[:-1])

    label_name = "lame" if row["label"] == 1 else "non_lame"

    # The path for cropped images is FRAMES_ROOT / label_name / video_folder_name / original_image_name
    original_image_path = os.path.join(FRAMES_ROOT, label_name, video_folder_name, original_image_name)

    # Add a check for file existence before proceeding
    if not os.path.exists(original_image_path):
        print(f"Warning: Original cropped image not found at {original_image_path}, skipping augmentation for {original_image_name}.")
        continue

    # Define output directory for augmented images, preserving video structure
    out_video_dir = os.path.join(OUT_ROOT, label_name, video_folder_name)
    os.makedirs(out_video_dir, exist_ok=True)

    flipped_image_name = f"flipped_{original_image_name}"
    output_img_path = os.path.join(out_video_dir, flipped_image_name)

    # Flip image and get new dimensions (if successful)
    new_width, new_height = flip_image(original_image_path, output_img_path)

    if new_width is not None and new_height is not None:
        # Augment keypoints
        augmented_row = augment_keypoints(row, KP_SWAP_MAP, df_original.columns)

        # Update the image name to the new flipped image name
        augmented_row["image"] = flipped_image_name
        augmented_data.append(augmented_row)
    else:
        # This else block should now only be reached if cv2.imread fails for an existing file
        print(f"Skipping augmentation for {original_image_name} due to image loading or processing error (path: {original_image_path}).")

df_augmented = pd.DataFrame(augmented_data)
print(f"Successfully augmented {len(df_augmented)} samples.")
print("Augmented DataFrame head:")
print(df_augmented.head())

**Reasoning**:
The previous steps successfully augmented the data by flipping images and their keypoints. Now, the `df_augmented` DataFrame contains the augmented data. The next logical step is to combine this augmented data with the original data (`df_original`) and save the complete dataset to a new CSV file. This will finalize the data augmentation process for keypoints.



In [ ]:
df_combined = pd.concat([df_original, df_augmented], ignore_index=True)

# Save the combined DataFrame to a new CSV file
output_csv_path = os.path.join(OUTPUT_CSV_DIR, "pose_combined_augmented.csv")
df_combined.to_csv(output_csv_path, index=False)

print(f"Successfully combined original and augmented data. Total samples: {len(df_combined)}")
print(f"Combined dataset saved to: {output_csv_path}")
print("Augmentation process completed successfully!")

## Final Task

### Subtask:
Confirm that the data augmentation process is complete, and both the `cow_crop_augmented` and `annotations_augmented` directories have been populated with the new images and keypoint CSVs.


## Summary:

### Q&A
The data augmentation process was successfully completed. Both the `cow_crop_augmented` directory (for new images) and the `annotations_augmented` directory (for new keypoint CSVs) have been populated. Specifically, augmented images were saved to `/content/drive/MyDrive/cow/2/dataset/cow_crop_augmented`, and the combined keypoint data was saved to `/content/drive/MyDrive/cow/2/dataset/annotations_augmented/pose_combined_augmented.csv`.

### Data Analysis Key Findings
*   The original `pose_lame.csv` and `pose_non_lame.csv` files were successfully loaded and combined into a single DataFrame, `df_original`, containing 500 samples.
*   A subset of 250 samples (50% of the total original samples) was selected for horizontal augmentation.
*   During the augmentation process, errors were encountered initially due to incorrect path construction and then due to missing original image files.
*   Robust error handling was implemented by checking for the existence of original image files before processing. This revealed that 4 out of the 250 selected samples for augmentation were skipped because their corresponding original cropped image files could not be found on disk.
*   A total of 246 samples were successfully augmented with horizontally flipped images and adjusted keypoints.
*   The original 500 samples and the 246 augmented samples were combined into a new DataFrame, `df_combined`, totaling 746 samples.
*   The combined dataset was saved as `pose_combined_augmented.csv` in the specified output directory.

### Insights or Next Steps
*   Investigate and resolve the issue of missing original cropped image files (4 out of 250 selected for augmentation were not found) to ensure data integrity and maximize the effectiveness of augmentation.
*   Consider performing a visual inspection of a sample of the augmented images and their corresponding keypoints to verify the accuracy of the horizontal flips and keypoint adjustments.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from math import atan2, acos

# ---------------------------
# Helper functions
# ---------------------------
def dist(p, q):
    return np.sqrt((p[0]-q[0])**2 + (p[1]-q[1])**2)

def angle(a, b, c):
    ba = np.array(a) - np.array(b)
    bc = np.array(c) - np.array(b)
    cosang = np.dot(ba, bc) / (np.linalg.norm(ba)*np.linalg.norm(bc) + 1e-8)
    return acos(np.clip(cosang, -1, 1))

def get_point(row, i):
    return (row[f"kp{i}_x"], row[f"kp{i}_y"])

# ---------------------------
# Keypoint index mapping
# ---------------------------
KP = {
    "head_muzzle": 0, "head_poll": 1, "withers": 2, "back": 3, "pin": 4,
    "FL_upper": 5, "FL_knee": 6, "FL_hoof": 7,
    "FR_upper": 8, "FR_knee": 9, "FR_hoof": 10,
    "HL_upper": 11, "HL_knee": 12, "HL_hoof": 13,
    "HR_upper": 14, "HR_knee": 15, "HR_hoof": 16
}

# ---------------------------
# Load CSV
# ---------------------------
CSV_PATH = "/content/drive/MyDrive/cow/2/dataset/annotations_augmented/pose_combined_augmented.csv"
df = pd.read_csv(CSV_PATH)

features = []

for _, row in df.iterrows():
    P = {i: get_point(row, i) for i in range(17)}

    # Body length
    body_len = dist(P[KP["withers"]], P[KP["pin"]])

    # Leg lengths
    FL = dist(P[KP["FL_upper"]], P[KP["FL_knee"]]) + dist(P[KP["FL_knee"]], P[KP["FL_hoof"]])
    FR = dist(P[KP["FR_upper"]], P[KP["FR_knee"]]) + dist(P[KP["FR_knee"]], P[KP["FR_hoof"]])
    HL = dist(P[KP["HL_upper"]], P[KP["HL_knee"]]) + dist(P[KP["HL_knee"]], P[KP["HL_hoof"]])
    HR = dist(P[KP["HR_upper"]], P[KP["HR_knee"]]) + dist(P[KP["HR_knee"]], P[KP["HR_hoof"]])

    # Angles
    spine_curv = angle(P[KP["withers"]], P[KP["back"]], P[KP["pin"]])
    FL_ang = angle(P[KP["FL_upper"]], P[KP["FL_knee"]], P[KP["FL_hoof"]])
    FR_ang = angle(P[KP["FR_upper"]], P[KP["FR_knee"]], P[KP["FR_hoof"]])
    HL_ang = angle(P[KP["HL_upper"]], P[KP["HL_knee"]], P[KP["HL_hoof"]])
    HR_ang = angle(P[KP["HR_upper"]], P[KP["HR_knee"]], P[KP["HR_hoof"]])

    hoof_y = [P[KP["FL_hoof"]][1], P[KP["FR_hoof"]][1],
              P[KP["HL_hoof"]][1], P[KP["HR_hoof"]][1]]

    try:
        hull = ConvexHull([P[KP["FL_hoof"]], P[KP["FR_hoof"]],
                           P[KP["HL_hoof"]], P[KP["HR_hoof"]]])
        support_area = hull.volume
    except:
        support_area = 0.0

    # ---------------------------
    # 🔹 NEW FEATURES (5 added)
    # ---------------------------

    # 1. Front step asymmetry
    front_step_diff = abs(P[KP["FL_hoof"]][0] - P[KP["FR_hoof"]][0]) / body_len

    # 2. Hind step asymmetry
    hind_step_diff = abs(P[KP["HL_hoof"]][0] - P[KP["HR_hoof"]][0]) / body_len

    # 3. Cross-support ratio
    cross_support_ratio = abs((FL + HR) - (FR + HL)) / (FL + FR + HL + HR + 1e-6)

    # 4. Head-to-spine ratio
    head_len = dist(P[KP["head_muzzle"]], P[KP["head_poll"]])
    spine_len = dist(P[KP["withers"]], P[KP["pin"]])
    head_to_spine_ratio = head_len / (spine_len + 1e-6)

    # 5. Stance width
    stance_width = (
        abs(P[KP["FL_hoof"]][0] - P[KP["FR_hoof"]][0]) +
        abs(P[KP["HL_hoof"]][0] - P[KP["HR_hoof"]][0])
    ) / (2 * body_len)

    # ---------------------------
    # Feature dictionary
    # ---------------------------
    feat = {
        "spine_curvature": spine_curv,
        "head_drop": abs(P[KP["head_muzzle"]][1] - P[KP["head_poll"]][1]) / body_len,
        "front_leg_asym": abs(FL - FR) / body_len,
        "hind_leg_asym": abs(HL - HR) / body_len,
        "diag_imbalance": abs((FL + HR) - (FR + HL)) / body_len,
        "front_flexion": (FL_ang + FR_ang) / 2,
        "hind_flexion": (HL_ang + HR_ang) / 2,
        "hoof_height_var": np.var(hoof_y),
        "support_area": support_area,
        "weight_bias": abs(np.mean(hoof_y[:2]) - np.mean(hoof_y[2:])) / body_len,
        "body_slope": atan2(P[KP["pin"]][1]-P[KP["withers"]][1],
                            P[KP["pin"]][0]-P[KP["withers"]][0]),
        "mean_leg_length": np.mean([FL, FR, HL, HR]),
        "leg_length_var": np.var([FL, FR, HL, HR]),
        "compactness": (body_len**2) / (support_area + 1e-6),

        # 🔹 Added features
        "front_step_diff": front_step_diff,
        "hind_step_diff": hind_step_diff,
        "cross_support_ratio": cross_support_ratio,
        "head_to_spine_ratio": head_to_spine_ratio,
        "stance_width": stance_width,

        "label": row["label"]
    }

    features.append(feat)

# ---------------------------
# Save engineered features
# ---------------------------
df_feat = pd.DataFrame(features)
OUT_PATH = "/content/drive/MyDrive/cow/2/dataset/annotations_augmented/features_engineered.csv"
df_feat.to_csv(OUT_PATH, index=False)

print("✅ Feature engineering completed (19 features)")
print("Saved to:", OUT_PATH)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load engineered features
CSV_PATH = "/content/drive/MyDrive/cow/2/dataset/annotations_augmented/features_engineered.csv"
df = pd.read_csv(CSV_PATH)

# Separate features and label
X = df.drop(columns=["label"])

# Compute correlation matrix
corr = X.corr()

# ---------------------------
# Heatmap
# ---------------------------
plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

# ---------------------------
# List highly correlated feature pairs
# ---------------------------
threshold = 0.85
high_corr = []

for i in range(len(corr.columns)):
    for j in range(i):
        if abs(corr.iloc[i, j]) > threshold:
            high_corr.append((
                corr.columns[i],
                corr.columns[j],
                corr.iloc[i, j]
            ))

print("Highly correlated feature pairs (|corr| > 0.85):")
for f1, f2, c in high_corr:
    print(f"{f1}  <->  {f2} : {c:.3f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Split
X = df.drop(columns=["label"])
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train model
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight="balanced"
)
rf.fit(X_train, y_train)

# Importance
importances = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

# Plot
plt.figure(figsize=(10, 6))
importances.plot(kind="bar")
plt.title("Random Forest Feature Importance")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

print(importances)

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

perm_imp = pd.Series(
    perm.importances_mean,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
perm_imp.plot(kind="bar")
plt.title("Permutation Feature Importance")
plt.ylabel("Decrease in Accuracy")
plt.tight_layout()
plt.show()

print(perm_imp)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
DATA_PATH = "/content/drive/MyDrive/cow/2/dataset/annotations_augmented/features_engineered.csv"
df = pd.read_csv(DATA_PATH)

y = df["label"]

In [ ]:
FEATURE_SETS = {
    "F19_All": [
        "spine_curvature","head_drop","front_leg_asym","hind_leg_asym",
        "diag_imbalance","front_flexion","hind_flexion","hoof_height_var",
        "support_area","weight_bias","body_slope","mean_leg_length",
        "leg_length_var","compactness",
        "front_step_diff","hind_step_diff",
        "cross_support_ratio","head_to_spine_ratio","stance_width"
    ],

    "F12_Selected": [
        "spine_curvature","head_to_spine_ratio","mean_leg_length",
        "leg_length_var","support_area","hoof_height_var",
        "stance_width","weight_bias","body_slope",
        "compactness","front_flexion","hind_flexion"
    ],

    "F10_Compact": [
        "spine_curvature","head_to_spine_ratio","mean_leg_length",
        "leg_length_var","support_area","hoof_height_var",
        "stance_width","weight_bias","body_slope","compactness"
    ],

    "F8_Lightweight": [
        "spine_curvature","head_to_spine_ratio","mean_leg_length",
        "leg_length_var","support_area","hoof_height_var",
        "stance_width","compactness"
    ]
}

In [ ]:
CLASSIFIERS = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300, random_state=42
    ),

    "SVM": SVC(
        kernel="rbf", probability=True, random_state=42
    ),

    "LogisticRegression": LogisticRegression(
        max_iter=1000, random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=7
    )
}

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    }

    print(classification_report(y_test, y_pred, digits=4))
    return metrics

In [ ]:
results = []

for feat_name, feat_cols in FEATURE_SETS.items():
    print("\n" + "="*70)
    print(f"🔹 Feature Set: {feat_name} ({len(feat_cols)} features)")
    print("="*70)

    X = df[feat_cols]

    # shuffle + split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        shuffle=True,
        random_state=42 # Changed from np.random.randint(0, 10000) to a fixed integer
    )

    for clf_name, clf in CLASSIFIERS.items():
        print("\n" + "-"*60)
        print(f"Classifier: {clf_name}")
        print("-"*60)

        # Scale when needed
        if clf_name in ["SVM", "LogisticRegression", "KNN"]:
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", clf)
            ])
        else:
            model = clf

        metrics = evaluate_model(
            model, X_train, X_test, y_train, y_test
        )

        metrics["Classifier"] = clf_name
        metrics["Feature_Set"] = feat_name
        results.append(metrics)

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df[
    ["Feature_Set","Classifier","Accuracy","Precision","Recall","F1","ROC_AUC"]
]

print("\n📊 FINAL SUMMARY")
print(results_df)